# Step 1 — Overview: Script description
# ------------------------------------
# This script creates a new CSV file called "energy_prices.csv" in the folder:
# /Users/jakegehrung/Desktop/data_raw/tariff
#
# It performs the following steps:
# 1. Generates a "Time" column with 17,520 rows (half-hour intervals over a full year),
#    starting at 00:30:00 and ending at 00:00:00 of the final day (excluded start at 00:00:00).
# 2. Reads multiple source CSV files from the same folder.
# 3. Extracts the second column from each file.
# 4. Renames each extracted column according to a specified naming convention.
# 5. Combines all columns into a single DataFrame in a strict column order.
# 6. Saves the final DataFrame as "energy_prices.csv".
#
# The script assumes all source CSVs contain at least two columns and exactly 17,520 rows.

In [1]:
# Step 2 — Import libraries and define paths
# ------------------------------------------
import pandas as pd
import os

# Base directory
base_path = "/Users/jakegehrung/Desktop/data_raw/tariff"

# Output file path
output_file = os.path.join(base_path, "energy_prices.csv")

In [2]:
# Step 3 — Create Time column (17,520 half-hour intervals starting at 00:30:00)
# -----------------------------------------------------------------------------
time_series = pd.date_range(
    start="00:30:00",
    periods=17520,
    freq="30min"
).time

# Convert to string format HH:MM:SS
time_column = [t.strftime("%H:%M:%S") for t in time_series]

# Create initial DataFrame
df = pd.DataFrame({"Time": time_column})

# Quick validation
assert len(df) == 17520, "Time column does not have 17,520 rows"

In [3]:
# Step 4 — Define source files and desired column order
# ----------------------------------------------------
file_mapping = {
    "flex_import": "flex_import.csv",
    "flex_export": "flex_export.csv",
    "fixed_import": "fixed_import.csv",
    "fixed_export": "fixed_export.csv",  # NOTE: This file must exist
    "fixed_coal": "fixed_coal.csv",
    "fixed_lpg": "fixed_lpg.csv",
    "fixed_mains_gas": "fixed_mains_gas.csv",
    "fixed_mineral_wood": "fixed_mineral_wood.csv",
    "fixed_oil": "fixed_oil.csv",
    "fixed_wood_chips": "fixed_wood_chips.csv",
    "fixed_wood_logs": "fixed_wood_logs.csv",
    "fixed_wood_pellets": "fixed_wood_pellets.csv"
}

# Desired column order
column_order = [
    "Time",
    "flex_import",
    "flex_export",
    "fixed_import",
    "fixed_export",
    "fixed_coal",
    "fixed_lpg",
    "fixed_mains_gas",
    "fixed_mineral_wood",
    "fixed_oil",
    "fixed_wood_chips",
    "fixed_wood_logs",
    "fixed_wood_pellets"
]

In [4]:
# Step 5 — Load each CSV and extract second column
# -----------------------------------------------
for col_name, file_name in file_mapping.items():
    file_path = os.path.join(base_path, file_name)
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_name}")
    
    temp_df = pd.read_csv(file_path)
    
    if temp_df.shape[1] < 2:
        raise ValueError(f"{file_name} does not have at least two columns")
    
    second_column = temp_df.iloc[:, 1]
    
    if len(second_column) != 17520:
        raise ValueError(f"{file_name} does not have 17,520 rows")
    
    df[col_name] = second_column.values

In [5]:
# Step 6 — Reorder columns and save final CSV
# -------------------------------------------
df = df[column_order]

# Save to CSV
df.to_csv(output_file, index=False)

print(f"File successfully created at: {output_file}")

File successfully created at: /Users/jakegehrung/Desktop/data_raw/tariff/energy_prices.csv
